# 2.2 Model interrogation

In [1]:
#__ Import: Numerical __
import numpy as np

#__ Import: Topophone2 __
from topophone2.topophone import decompose
from topophone2.inversion import MarginalStats
from topophone2.utils.read import read_txt

#__ Import: Plotting __
import matplotlib.pyplot as plt
import matplotlib.tri as tri
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import cmcrameri.cm as cmc

#__ Set up: Plotting __
theme = ['#DC4424', '#74A466', '#24448C', '#F8BC5C']
small, large = 8, 10
plt.rcParams["font.sans-serif"] = ['Arial']

#__ Set up: Random seed __
np.random.seed(2026)

Hello from topophone2!


### 2.2.1 Load example data

In [2]:
#__ Read: Signal + noise __
out = read_txt("../data/hukkakero/signal_topophone2.txt")
freq_Hz   = out[0]  # Frequency vector         [Hz]       (frequency)
x         = out[1]  # Receiver position        [m]        (receiver, direction)
Gamma_obs = out[2]  # Array-covariance matrix  [Pa2.Hz-1] (frequency, receiver, receiver)
N_obs     = out[3]  # Independent observations
freq      = 2 * np.pi * freq_Hz  # Angular frequency vector [rad.s-1] (frequency)

#__ Read: Noise only __
out = read_txt("../data/hukkakero/noise_topophone2.txt")
Gamma_obs_noise = out[2]  # Array-covariance matrix  [Pa2.Hz-1] (frequency, receiver, receiver)
N_obs_noise     = out[3]  # Independent observations

### 2.2.2 Invert example data

Set inversion parameters...

In [3]:
#__ Signals present __
sigs = [
    'skewnormal',                     # Skew-normal density function
    'isoselfnoise',                   # Self-noise
    {'sig':'noise2D', 'max_order':2}  # Cylindrical-harmonic noise field
]

#__ Prior bounds __
lower = np.array([                                      # Lower bound of prior pdf
    -40., -2.7e-3, 1.0e-3, 1e-6, 1e-6, -0.9, -5., -5.,  #     Skew-normal parameters [weight, xi_x, xi_y, sigma_x, sigma_y, rho_xy, alpha_x, alpha_y]
    -55.,                                               #     Self-noise parameters  [weight]
    -55., 2.2e-3, 0.1, -0.35, -0.55, -0.35              #     Noise field parameters [weight, slowness, a_1, a_2, b_1, b_2]
])
upper = np.array([                                      # Upper bound of prior pdf
    -30., -2.2e-3, 1.6e-3, 1e-3, 1e-3,  0.9,  5.,  5.,
    -45.,
    -45., 2.5e-3, 0.3, -0.15, -0.30, -0.15
])

#__ Number of flows __
N_flow = 8

#__ Inversion parameters __
learning_rate = 5e-4                       # Learning rate for the optimiser
N_iterations  = 30000                      # Number of iterations for optimiser
N_montecarlo  = 50                         # Number of samples to estimate ELBO

#__ Number of samples to draw __
N_samples = 20000

In [4]:
#__ Perform inversion __
decompose(
    Gamma_obs       = Gamma_obs,
    N_obs           = N_obs,
    Gamma_obs_noise = Gamma_obs_noise,
    N_obs_noise     = N_obs_noise,
    x               = x,
    freq            = freq,
    sigs            = sigs,
    prior_bounds    = (lower,upper),
    N_flow          = N_flow,
    learning_rate   = learning_rate,
    outpath         = "../models/hukkakero",
    N_iterations    = N_iterations,
    N_montecarlo    = N_montecarlo,
    N_out           = 6,
    verbose         = True,
    N_samples       = N_samples
)

#__ Read results __
m = np.loadtxt(f"../models/hukkakero/model-samples_iteration-{N_iterations:05d}.txt")

----------------------------------------

Iteration:     0,	Loss:   -1712.13
The elapsed time is: 0.06 s
Iteration:  5000,	Loss:  -22310.14
The elapsed time is: 198.24 s
Iteration: 10000,	Loss:  -22318.11
The elapsed time is: 398.67 s
Iteration: 15000,	Loss:  -22318.41
The elapsed time is: 599.54 s
Iteration: 20000,	Loss:  -22318.91
The elapsed time is: 800.97 s
Iteration: 25000,	Loss:  -22319.03
The elapsed time is: 1002.27 s
Iteration: 30000,	Loss:  -22319.09
The elapsed time is: 1202.07 s
----------------------------------------

Finish training!


In [ ]:
#__ Skew-normal parser __
def skewnormal_params(
    xi    : np.ndarray,
    sigma : np.ndarray,
    corr  : np.ndarray,
    alpha : np.ndarray
):
    """
    Construct skewnormal parameters from model parameters.

    Input
    -----
    xi : ``numpy.ndarray``
        Location vector
        (model, direction)

    sigma : ``numpy.ndarray``
        Scale vector
        (model, direction)

    corr : ``numpy.ndarray``
        Correlation matrix off-diagonal
        (model, direction pair)

    alpha : ``numpy.ndarray``
        Skewness vector
        (model, direction)
    """

    #__ np.einsum subscripts:
    #__ [i] model
    #__ [l],[p] direction

    #__ Construct correlation matrix __
    N_dir              = xi.shape[1]
    _iu0_dir, _iu1_dir = np.triu_indices(N_dir, k=1)
    Omega_z = np.ones((xi.shape[0], N_dir, N_dir), dtype=float)
    Omega_z[:,_iu0_dir,_iu1_dir] = corr
    Omega_z[:,_iu1_dir,_iu0_dir] = corr

    #__ Calculate skewness-direction vector (model, direction) __
    Oza   = np.einsum('ilp,ip->il', Omega_z, alpha, optimize=True)
    N     = 1.0 / np.sqrt(1.0 + np.einsum('il,il->i', alpha, Oza, optimize=True))
    delta = N[:,None] * Oza

    #__ Scale skewness-direction vector __
    wd = sigma * delta

    #__ Calculate mean __
    mean = xi + wd * np.sqrt(2/np.pi)

    #__ Project to cylindrical/spherical coordinates __
    xi_circ   = 
    wd_circ   = 
    mean_circ = 

    #__ Return output __
    return wd

In [ ]:
import numpy as np
from scipy.stats import norm
import contourpy

#__ Skew-normal parser __
def skewnormal_params(
    xi           : np.ndarray,
    sigma        : np.ndarray,
    corr         : np.ndarray,
    alpha        : np.ndarray,
    contour_mass : list  = None,
    N_grid       : int   = 201,
    n_scale      : float = 5.0
):
    """
    Construct skew-normal summary quantities from model parameters, in Cartesian and
    cylindrical/spherical coordinates, with optional highest-density contours (2D only).

    Input
    -----
    xi : ``numpy.ndarray``
        Location vector [s.m-1], components (East, North[, Up])
        (model, direction)

    sigma : ``numpy.ndarray``
        Scale vector [s.m-1]
        (model, direction)

    corr : ``numpy.ndarray``
        Correlation matrix off-diagonal (upper-triangle order)
        (model, direction pair)

    alpha : ``numpy.ndarray``
        Skewness (shape) vector
        (model, direction)

    contour_mass : ``list`` (optional)
        Probability masses enclosed by highest-density contours, e.g. ``[0.5, 0.9]`` (2D only)

    N_grid : ``int`` (default ``201``)
        Number of grid points per direction for contouring

    n_scale : ``float`` (default ``5.0``)
        Half-width of contouring grid in units of the largest scale

    Output
    ------
    out : ``dict``
        ``delta``     : skewness-direction vector (model, direction)
        ``wd``        : scaled skewness-direction vector W delta [s.m-1] (model, direction)
        ``mean``      : mean vector [s.m-1] (model, direction)
        ``xi_circ``   : location as (r [s.m-1], phi [deg][, theta [deg]]) (model, coordinate)
        ``mean_circ`` : mean as (r [s.m-1], phi [deg][, theta [deg]]) (model, coordinate)
        ``wd_circ``   : W delta in the local basis at xi, as linearised changes
                        (dr [s.m-1], dphi [deg][, dtheta [deg]]) (model, coordinate);
                        dphi > 0 is clockwise, dtheta > 0 is upwards
        ``contours``  : (if requested) list over models of dicts {mass: [vertices (N, 2), ...]}

    Note
    ----
    phi is measured clockwise from North and theta upwards from the horizontal, for the
    vector itself (i.e. the propagation direction if xi is a propagation slowness).
    Backazimuth is (phi + 180) mod 360.
    """

    #__ np.einsum subscripts:
    #__ [i] model
    #__ [l],[p] direction

    #__ Construct correlation matrix __
    N_model, N_dir     = xi.shape
    _iu0_dir, _iu1_dir = np.triu_indices(N_dir, k=1)
    Omega_z = np.ones((N_model, N_dir, N_dir), dtype=float)
    Omega_z[:,_iu0_dir,_iu1_dir] = corr
    Omega_z[:,_iu1_dir,_iu0_dir] = corr

    #__ Calculate skewness-direction vector (model, direction) __
    Oza   = np.einsum('ilp,ip->il', Omega_z, alpha, optimize=True)
    N     = 1.0 / np.sqrt(1.0 + np.einsum('il,il->i', alpha, Oza, optimize=True))
    delta = N[:,None] * Oza

    #__ Scale skewness-direction vector __
    wd = sigma * delta

    #__ Calculate mean __
    mean = xi + wd * np.sqrt(2/np.pi)

    #__ Cartesian -> cylindrical/spherical (r, phi clockwise from North, theta up from horizontal) __
    def to_circ(v):
        r   = np.linalg.norm(v, axis=1)
        phi = np.degrees(np.arctan2(v[:,0], v[:,1])) % 360
        if (N_dir == 2): return np.stack([r, phi], axis=1)
        theta = np.degrees(np.arctan2(v[:,2], np.hypot(v[:,0], v[:,1])))
        return np.stack([r, phi, theta], axis=1)

    #__ Project to cylindrical/spherical coordinates __
    xi_circ   = to_circ(xi)
    mean_circ = to_circ(mean)

    #__ Local orthonormal basis at xi: e_r, e_phi (clockwise), [e_theta (upwards)] __
    r     = xi_circ[:,0]
    phi   = np.radians(xi_circ[:,1])
    theta = np.radians(xi_circ[:,2]) if (N_dir == 3) else np.zeros(N_model)
    ct, st, cp, sp = np.cos(theta), np.sin(theta), np.cos(phi), np.sin(phi)
    if (N_dir == 2):
        e_r   = np.stack([sp, cp], axis=1)
        e_phi = np.stack([cp, -sp], axis=1)
    else:
        e_r     = np.stack([ct*sp, ct*cp, st], axis=1)
        e_phi   = np.stack([cp, -sp, np.zeros(N_model)], axis=1)
        e_theta = np.stack([-st*sp, -st*cp, ct], axis=1)

    #__ W delta as linearised changes in (r, phi, theta) __
    dr   = np.sum(wd * e_r, axis=1)
    dphi = np.degrees(np.sum(wd * e_phi, axis=1) / (r * ct))
    if (N_dir == 2): wd_circ = np.stack([dr, dphi], axis=1)
    else:
        dtheta  = np.degrees(np.sum(wd * e_theta, axis=1) / r)
        wd_circ = np.stack([dr, dphi, dtheta], axis=1)

    #__ Output __
    out = dict(delta=delta, wd=wd, mean=mean,
               xi_circ=xi_circ, mean_circ=mean_circ, wd_circ=wd_circ)

    #__ Highest-density contours (2D only) __
    if (contour_mass is not None):
        if (N_dir != 2): raise ValueError('Contours are only available in 2D.')
        out['contours'] = [
            _skewnormal_contours(xi[i], sigma[i], Omega_z[i], alpha[i],
                                 contour_mass, N_grid, n_scale)
            for i in range(N_model)
        ]

    #__ Return output __
    return out

#__ Highest-density contours of one 2D skew-normal __
def _skewnormal_contours(xi, sigma, Omega_z, alpha, contour_mass, N_grid, n_scale):

    #__ Grid centred on xi, spanning n_scale times the largest scale __
    h  = n_scale * sigma.max()
    g0 = np.linspace(xi[0] - h, xi[0] + h, N_grid)
    g1 = np.linspace(xi[1] - h, xi[1] + h, N_grid)
    P0, P1 = np.meshgrid(g0, g1, indexing='xy')
    Y  = np.stack([P0 - xi[0], P1 - xi[1]], axis=-1)       # (grid, grid, direction)

    #__ Density: 2 phi_2(y; Omega) Phi(alpha' W^-1 y), Omega = W Omega_z W __
    Omega     = sigma[:,None] * Omega_z * sigma[None,:]
    Omega_inv = np.linalg.inv(Omega)
    quad      = np.einsum('abl,lp,abp->ab', Y, Omega_inv, Y)
    pdf       = (2.0 * np.exp(-0.5*quad) / (2*np.pi*np.sqrt(np.linalg.det(Omega)))
                 * norm.cdf(Y @ (alpha / sigma)))

    #__ Density thresholds enclosing requested masses __
    dA     = (g0[1] - g0[0]) * (g1[1] - g1[0])
    z_sort = np.sort(pdf.ravel())[::-1]
    cum    = np.cumsum(z_sort) * dA
    cum   /= cum[-1]                                        # Normalise out grid truncation
    levels = {mass: z_sort[np.searchsorted(cum, mass)] for mass in contour_mass}

    #__ Extract contour lines __
    gen = contourpy.contour_generator(g0, g1, pdf)
    return {mass: gen.lines(level) for mass, level in levels.items()}

In [ ]:
import matplotlib.pyplot as plt

#__ Plot ensemble of skew-normal contours from posterior samples __
def plot_skewnormal_ensemble(m, N_draw=100, mass=0.5, polar=False, ref=None, ax=None, seed=0):

    #__ Draw a random subset of posterior samples __
    rng = np.random.default_rng(seed)
    idx = rng.choice(m.shape[0], size=min(N_draw, m.shape[0]), replace=False)
    xi, sig, corr, alpha = m[idx,1:3], m[idx,3:5], m[idx,5:6], m[idx,6:8]
    out = skewnormal_params(xi, sig, corr, alpha, contour_mass=[mass])

    #__ Coordinate transform: slowness [s.km-1] or (backazimuth [deg], velocity [m.s-1]) __
    def tf(p):
        p = np.atleast_2d(p)
        if not polar: return 1e3 * p
        return np.stack([(np.degrees(np.arctan2(p[:,0], p[:,1])) + 180) % 360,
                         1 / np.linalg.norm(p, axis=1)], axis=1)

    if ax is None: fig, ax = plt.subplots(figsize=(6, 6))

    #__ Contours and W delta segment for each draw __
    for i in range(len(idx)):
        for L in out['contours'][i][mass]:
            v = tf(L) ; ax.plot(v[:,0], v[:,1], color='C0', lw=0.6, alpha=0.25)
        seg = tf(np.stack([xi[i], xi[i] + out['wd'][i]]))
        ax.plot(seg[:,0], seg[:,1], color='C3', lw=0.8, alpha=0.4)

    #__ Locations and means __
    loc = tf(xi) ; mu = tf(out['mean'])
    ax.scatter(loc[:,0], loc[:,1], s=4, color='k',  alpha=0.5, label=r'location $\xi$', zorder=3)
    ax.scatter(mu[:,0],  mu[:,1],  s=4, color='C3', alpha=0.5, label='mean', zorder=3)

    #__ Reference (best beam) given as (backazimuth [deg], velocity [m.s-1]) __
    if ref is not None:
        baz, v = ref ; ph = np.radians(baz + 180)
        r_pt = tf(np.array([np.sin(ph), np.cos(ph)]) / v)[0]
        ax.plot(*r_pt, marker='*', ms=12, color='C2', mec='k', label='best beam', zorder=4)

    #__ Axes __
    if polar: ax.set_xlabel('Backazimuth [deg]') ; ax.set_ylabel(r'Apparent velocity [m.s$^{-1}$]')
    else:     ax.set_xlabel(r'$p_E$ [s.km$^{-1}$]') ; ax.set_ylabel(r'$p_N$ [s.km$^{-1}$]') ; ax.set_aspect('equal')
    ax.set_title(f'{int(100*mass)}% highest-density contours ({len(idx)} posterior draws)', fontsize=10)
    ax.legend(fontsize=8, loc='best')
    return ax

In [ ]:
#__ Model vector layout __
#   [0]  w_sn      [1-2] xi_E, xi_N   [3-4] sigma_E, sigma_N   [5] rho   [6-7] alpha_E, alpha_N
#   [8]  w_self
#   [9]  w_noise   [10]  s            [11-12] a_1, a_2         [13-14] b_1, b_2
labels = [r'$w_\mathrm{sn}$ [dB]', r'$\xi_E$ [s.km$^{-1}$]', r'$\xi_N$ [s.km$^{-1}$]',
          r'$\sigma_E$ [s.km$^{-1}$]', r'$\sigma_N$ [s.km$^{-1}$]', r'$\rho$',
          r'$\alpha_E$', r'$\alpha_N$',
          r'$w_\mathrm{self}$ [dB]',
          r'$w_\mathrm{noise}$ [dB]', r'$s$ [s.km$^{-1}$]',
          r'$a_1$', r'$a_2$', r'$b_1$', r'$b_2$']
scale = np.ones(m.shape[1]) ; scale[[1, 2, 3, 4, 10]] = 1e3   # Slownesses in s.km-1

#__ Best-beam reference __
baz_bb, v_bb = 112.9, 422.6

#__ Set up figure __
fig, axes = plt.subplots(3, 6, figsize=(18, 8), constrained_layout=True)
axes = axes.ravel()

#__ Helper: histogram with median and 95% credible interval __
def hist_panel(ax, v, label, lo=None, hi=None, ref=None):
    rng_ = (lo, hi) if (lo is not None) else None
    ax.hist(v, bins=50, range=rng_, density=True, color='C0', alpha=0.8)
    if (lo is not None):
        ax.axvline(lo, color='k', ls='--', lw=1) ; ax.axvline(hi, color='k', ls='--', lw=1)
    q025, q50, q975 = np.percentile(v, [2.5, 50, 97.5])
    ax.axvline(q50, color='C3', lw=1.5)
    ax.axvspan(q025, q975, color='C3', alpha=0.15)
    if (ref is not None): ax.axvline(ref, color='C2', lw=1.5, ls=':')
    ax.set_xlabel(label) ; ax.set_yticks([])
    ax.set_title(f'{q50:.3g} [{q025:.3g}, {q975:.3g}]', fontsize=9)

#__ Parameter marginals __
for k in range(m.shape[1]):
    hist_panel(axes[k], m[:, k]*scale[k], labels[k], lower[k]*scale[k], upper[k]*scale[k])

#__ Positivity of noise azimuthal density __
theta   = np.linspace(0, 2*np.pi, 361)
basis   = np.stack([np.cos(theta), np.cos(2*theta), np.sin(theta), np.sin(2*theta)])
rho_min = (1 + 2 * m[:, 11:15] @ basis).min(axis=1)
ax = axes[15]
ax.hist(rho_min, bins=50, density=True, color='C2', alpha=0.8)
ax.axvline(0, color='k', ls='--', lw=1)
ax.set_xlabel(r'$\min_\theta\, 2\pi\rho(\theta)$') ; ax.set_yticks([])
ax.set_title(f'{100*np.mean(rho_min < 0):.1f}% invalid', fontsize=9)

#__ Skew-normal mean: xi + W delta sqrt(2/pi), delta = Omega_z alpha / sqrt(1 + alpha' Omega_z alpha) __
xi, sig, rho_c, alpha = m[:, 1:3], m[:, 3:5], m[:, 5], m[:, 6:8]
Oza   = np.stack([alpha[:,0] + rho_c*alpha[:,1], rho_c*alpha[:,0] + alpha[:,1]], axis=1)
delta = Oza / np.sqrt(1 + np.sum(alpha*Oza, axis=1))[:, None]
p_mean = xi + sig * delta * np.sqrt(2/np.pi)

#__ Backazimuth and apparent velocity of the mean __
baz = np.degrees(np.arctan2(-p_mean[:,0], -p_mean[:,1])) % 360
vel = 1 / np.linalg.norm(p_mean, axis=1)
hist_panel(axes[16], baz, 'Backazimuth of mean [deg]', ref=baz_bb)
hist_panel(axes[17], vel, r'Apparent velocity of mean [m.s$^{-1}$]', ref=v_bb)

plt.show()

### 2.2.3 Interrogate model ensemble

In [ ]:
xi, sig, rho_c, alpha = m[:, 1:3], m[:, 3:5], m[:, 5], m[:, 6:8]

#__ delta = Omega_z alpha / sqrt(1 + alpha' Omega_z alpha) __
Oza   = np.stack([alpha[:,0] + rho_c*alpha[:,1], rho_c*alpha[:,0] + alpha[:,1]], axis=1)
delta = Oza / np.sqrt(1 + np.sum(alpha*Oza, axis=1))[:, None]

#__ Skew direction in slowness space [s.m-1] __
d = sig * delta

#__ Tangential (clockwise) and radial unit vectors at xi __
xi_abs = np.linalg.norm(xi, axis=1)[:, None]
t_cw   = np.stack([xi[:,1], -xi[:,0]], axis=1) / xi_abs
r_hat  = xi / xi_abs

#__ Components of the skew __
d_t = np.sum(d * t_cw, axis=1)   # > 0: clockwise
d_r = np.sum(d * r_hat, axis=1)  # > 0: towards slower apparent velocity

#__ Angular offset of the mean from xi [deg] __
dphi = np.degrees(np.sqrt(2/np.pi) * d_t / xi_abs[:, 0])

print(f'P(anti-clockwise) = {100*np.mean(d_t < 0):.3f} %')
print(f'mean offset from xi: {np.median(dphi):.2f} deg [{np.percentile(dphi,2.5):.2f}, {np.percentile(dphi,97.5):.2f}]')